# Feature Validation & Selection

## Objective

Evaluate engineered and original features to identify the most informative variables for churn prediction.

In [ ]:
# Local Jupyter / GitHub compatibility (no-op on Databricks)
import os
import sys
from pathlib import Path

if "DATABRICKS_RUNTIME_VERSION" not in os.environ:
    _root = next(
        (
            p
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "utils" / "databricks_compat.py").exists()
        ),
        None,
    )
    if _root and str(_root / "src") not in sys.path:
        sys.path.insert(0, str(_root / "src"))
    from utils.databricks_compat import inject_globals

    inject_globals(globals())


In [ ]:

from pyspark.sql import functions as F

In [ ]:

gold_df = spark.table(
    "workspace.default.customer_churn_gold"
)

display(gold_df.limit(5))

In [ ]:

print("=" * 60)
print(" GOLD DATASET")
print("=" * 60)

print(f"Rows    : {gold_df.count()}")
print(f"Columns : {len(gold_df.columns)}")

In [ ]:

family_churn = (
    gold_df
    .groupBy("HasFamily")
    .agg(
        F.count("*").alias("Customers"),
        F.sum(
            F.when(F.col("Churn") == "Yes",1)
            .otherwise(0)
        ).alias("Churned")
    )
    .withColumn(
        "Churn_Rate",
        F.round(
            (F.col("Churned")/F.col("Customers"))*100,
            2
        )
    )
)

display(family_churn)


## Business Insight

### Question

Does having a family reduce customer churn?

### Evidence

Customers with family: 19.82% churn

Customers without family: 34.24% churn

### Insight

Customers who have a partner or dependents are substantially more likely to stay with the company.

### Business Impact

ðŸ”´ High

### Recommendation

Target customers without family through retention campaigns, loyalty benefits, and personalized offers.

In [ ]:

services_summary = (
    gold_df
    .groupBy("TotalServicesSubscribed")
    .agg(
        F.count("*").alias("Customers"),
        F.sum(
            F.when(F.col("Churn")=="Yes",1)
            .otherwise(0)
        ).alias("Churned")
    )
    .withColumn(
        "Churn_Rate",
        F.round(
            (F.col("Churned")/F.col("Customers"))*100,
            2
        )
    )
    .orderBy("TotalServicesSubscribed")
)

display(services_summary)